# Objective 1: build the 10-K dataset (laptop steps)

Run the cells from top to bottom. Use the **.venv** kernel (top right).

This notebook covers the steps that run on the laptop. Each step reads files and writes files, so you can close the notebook between steps and pick up later.

## Step 1.1: download the 24 filings from SEC EDGAR

- **The 8 training companies:** their FY2024 and FY2025 10-Ks.
- **The 8 unseen companies** (CRWV, FIG, ORCL, AMD, FFIV, AKAM, SWKS, JKHY): their FY2025 10-K only. FFIV, AKAM, SWKS and JKHY were added on Nov 26.
- **The fiscal year** is the calendar year in which the filing's period ends, so NVIDIA's FY2025 is the year ending 2025-01-26. 10-K/A amendments are skipped.
- **Each filing** goes to `data/raw/<TICKER>_FY<year>/`, together with a `metadata.json` file.
- **`data/filings.csv`** lists all 24 filings with their accession numbers (the SEC's ID for each filing), so anyone can re-download the same ones.

In [ ]:
# Setup: find the project folder, load config.yaml, and tell EDGAR who you are.
import csv
import json
import os
import time
from pathlib import Path

import yaml
from edgar import Company, set_identity

# The notebook may run from the notebooks/ folder, so walk up until config.yaml is found.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
MANIFEST = ROOT / "data" / "filings.csv"

config = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))

identity = os.environ.get("SEC_IDENTITY")
assert identity, "Set the SEC_IDENTITY environment variable, then restart VS Code."
set_identity(identity)  # the SEC requires a name and email with every request

print("Project folder:", ROOT)

In [ ]:
# The list of (ticker, fiscal year) pairs to download.
unseen = config["heldout_companies"] + config["test_only_companies"]
training = [c for c in config["companies"] if c not in unseen]

pairs = [(t, fy) for t in training for fy in config["fiscal_years"]]  # 8 companies x 2 years
pairs += [(t, config["test_fy"]) for t in unseen]                   # 8 companies x FY2025

print(len(pairs), "filings:", pairs)

In [ ]:
# Two functions: find the right 10-K, and save it.

def find_10k(ticker, fiscal_year):
    """The single original 10-K whose period ends in `fiscal_year`."""
    filings = Company(ticker).get_filings(form="10-K", amendments=False)
    matches = [f for f in filings
               if f.form == "10-K" and str(f.period_of_report)[:4] == str(fiscal_year)]
    if len(matches) != 1:
        periods = [str(f.period_of_report) for f in filings]
        raise RuntimeError(f"{ticker} FY{fiscal_year}: expected 1 10-K, found {len(matches)}; periods {periods}")
    return matches[0]


def download(ticker, fiscal_year, force=False):
    """Save the main 10-K document, any Exhibit 13, and metadata.json. Returns the metadata."""
    folder = RAW / f"{ticker}_FY{fiscal_year}"
    meta_path = folder / "metadata.json"
    if meta_path.exists() and not force:
        print(f"{ticker} FY{fiscal_year}: already downloaded")
        return json.loads(meta_path.read_text(encoding="utf-8"))

    filing = find_10k(ticker, fiscal_year)
    folder.mkdir(parents=True, exist_ok=True)

    # The main 10-K document
    main_name = filing.primary_documents[0].document
    (folder / main_name).write_text(filing.html(), encoding="utf-8")

    # Exhibit 13: only if the company put its financial statements there instead of Item 8
    exhibit_13 = [a for a in filing.exhibits if (a.document_type or "").upper().startswith("EX-13")]
    for a in exhibit_13:
        content = a.download()
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="replace")
        (folder / a.document).write_text(content, encoding="utf-8")

    meta = {
        "ticker": ticker,
        "fiscal_year": fiscal_year,
        "form": filing.form,
        "company": filing.company,
        "cik": filing.cik,
        "period_of_report": str(filing.period_of_report),
        "filing_date": str(filing.filing_date),
        "accession_number": filing.accession_no,
        "url": filing.homepage_url,
        "main_document": main_name,
        "exhibit_13": [a.document for a in exhibit_13],
        "exhibits": [{"type": a.document_type or "", "document": a.document} for a in filing.exhibits],
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"{ticker} FY{fiscal_year}: saved {main_name} (period {meta['period_of_report']}, filed {meta['filing_date']})")
    return meta

In [ ]:
# Download all 24. Takes about a minute; filings already saved are skipped unless FORCE = True.
# FORCE = True downloads again even if a filing is already saved.
FORCE = False

rows, failures = [], []
for ticker, fy in pairs:
    try:
        rows.append(download(ticker, fy, force=FORCE))
    except Exception as error:  # keep going, and report at the end
        failures.append(f"{ticker} FY{fy}: {error}")
        print(f"{ticker} FY{fy}: FAILED: {error}")
    time.sleep(0.5)  # stay well under the SEC's limit of 10 requests per second

print(f"\n{len(rows)} downloaded, {len(failures)} failed")
for f in failures:
    print("  ", f)

In [ ]:
# Write data/filings.csv: the list of filings, which goes into git.
fields = ["ticker", "fiscal_year", "company", "period_of_report", "filing_date",
          "accession_number", "main_document", "exhibit_13", "url"]
with MANIFEST.open("w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow({**row, "exhibit_13": ";".join(row["exhibit_13"])})

print("Wrote", MANIFEST.relative_to(ROOT), "with", len(rows), "filings")

In [ ]:
# Check: is each file a complete 10-K? Looks for the main Items and any Exhibit 13.
import html
import re

for row in rows:
    folder = RAW / f"{row['ticker']}_FY{row['fiscal_year']}"
    raw_html = (folder / row["main_document"]).read_text(encoding="utf-8")
    text = html.unescape(re.sub(r"<[^>]+>", " ", raw_html))  # strip the HTML tags
    text = re.sub(r"\s+", " ", text)
    items = [i for i in ["1A", "1C", "7", "7A", "8", "9A"]
             if re.search(r"Item\s*" + i + r"\b", text, re.I)]
    print(f"{folder.name:13} {len(text.split()) / 1000:4.0f}k words   Items found: {', '.join(items):22} Exhibit 13: {row['exhibit_13'] or 'none'}")

## Step 1.2: split each filing into its Items

- **What an Item is:** a part of the 10-K that the SEC requires, such as Item 1A Risk Factors, Item 7 MD&A or Item 8 Financial Statements. Items are the top level of every table of contents we build.
- **How:** `edgartools` finds where each Item starts, using the filing's own table of contents, and gives each Item back as Markdown. In Markdown, `**text**` means bold, so headings show up as `**bold**` lines on their own.
- **Kept:** Items 1 to 14. Item 15 is dropped, because it's mainly the exhibit index (a list of attached documents), and so is any Item under 50 words, such as "Item 4. Mine Safety Disclosures: Not applicable."
- **The exception (Nov 26):** some companies, such as NVDA and ORCL, file their financial statements under Item 15, and their Item 8 is one sentence pointing there. When Item 8 has under 100 words, Item 15 is kept, under its own name, because that's where the statements are.
- **Saved to:** `data/parsed/<TICKER>_FY<year>/items.jsonl`, one line per Item. This folder isn't committed to git, because this notebook can rebuild it.
- **Headings set by size (Nov 26):** most filings put headings in bold, but FIG makes them bigger instead (24pt and 16pt, against 10pt body text). So before parsing, any text at 16pt or more is marked bold. This changes only FIG; the other 23 filings come out exactly the same.
- **Safety check:** step 1.3 finds the headings inside each Item from those bold lines. So for every Item we count the bold heading lines. A long Item with almost none means its headings were lost, for example because they're underlined rather than bold, and that filing then gets split by hand from its HTML instead.


In [ ]:
# Two functions: split one filing into its Items, and count heading lines.
import re

import pandas as pd
from edgar.documents import ParserConfig, parse_html

PARSED = ROOT / "data" / "parsed"
MIN_WORDS = 50          # Items shorter than this are dropped ("Not applicable")
DROP_ITEMS = {"15"}     # Item 15 is mainly the exhibit index...
POINTER_WORDS = 100     # ...unless Item 8 is shorter than this: then Item 8 only points to Item 15, which holds the statements

HEADING_PT = 16        # text at least this big (in points) counts as a heading, even if it isn't bold

# A heading line: a whole line in bold, e.g. "**Liquidity and Capital Resources**"
HEADING_LINE = re.compile(r"^\*\*[^*\n]{2,150}\*\*$")


def count_headings(markdown):
    lines = [line.strip() for line in markdown.splitlines() if line.strip()]
    return sum(1 for line in lines[1:] if HEADING_LINE.match(line))  # skip the Item's own title


def bold_large_text(html_text):
    """Mark text at HEADING_PT or bigger as bold, so headings set by size come out as **bold** lines."""
    def fix(match):
        style = match.group(1)
        size = re.search(r"font-size:\s*([\d.]+)pt", style)
        if size and float(size.group(1)) >= HEADING_PT:
            if "font-weight" in style:
                style = re.sub(r"font-weight:\s*\d+", "font-weight:700", style)
            else:
                style += ";font-weight:700"
        return f'style="{style}"'
    return re.sub(r'style="([^"]*)"', fix, html_text)


def split_items(folder):
    """Split one saved filing into Items. Returns a list of dicts, one per kept Item."""
    meta = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
    html_text = (folder / meta["main_document"]).read_text(encoding="utf-8")
    doc = parse_html(bold_large_text(html_text), ParserConfig(form="10-K"))

    sections = []
    for key, section in doc.sections.items():           # keys look like "part_ii_item_7a"
        item = key.split("_item_")[-1].upper()             # -> "7A"
        markdown = section.markdown() if callable(section.markdown) else section.markdown
        sections.append((key, item, section, markdown, len(markdown.split())))

    # If Item 8 is only a pointer (NVDA, ORCL), the financial statements are in Item 15, so keep it.
    item8_words = sum(words for _, item, _, _, words in sections if item == "8")
    drop = DROP_ITEMS - {"15"} if item8_words < POINTER_WORDS else DROP_ITEMS

    items = []
    for key, item, section, markdown, words in sections:
        if item in drop or words < MIN_WORDS:
            continue
        items.append({
            "item": item,
            "part": key.split("_item_")[0].replace("part_", "").upper(),   # "II"
            "title": section.title,
            "words": words,
            "bold_headings": count_headings(markdown),
            "confidence": section.confidence,
            "method": section.detection_method,
            "markdown": markdown,
        })
    return items

In [ ]:
# Split all 24 filings and save each one's Items. Takes a few minutes.
summary = []
for folder in sorted(RAW.glob("*_FY*")):
    items = split_items(folder)
    out_folder = PARSED / folder.name
    out_folder.mkdir(parents=True, exist_ok=True)
    with (out_folder / "items.jsonl").open("w", encoding="utf-8") as out:
        for it in items:
            out.write(json.dumps(it) + "\n")
    for it in items:
        summary.append({"filing": folder.name, **{k: v for k, v in it.items() if k != "markdown"}})
    kept_15 = "  (Item 15 kept: it holds the financial statements)" if any(it["item"] == "15" for it in items) else ""
    print(f"{folder.name:13} {len(items):2} Items kept{kept_15}")

summary = pd.DataFrame(summary)

In [ ]:
# Check 1: words in each Item, one row per filing. A blank means the Item wasn't found or was dropped.
order = ["1", "1A", "1B", "1C", "2", "3", "4", "5", "6", "7", "7A", "8", "9", "9A", "9B", "9C", "10", "11", "12", "13", "14", "15"]
words = summary.pivot(index="filing", columns="item", values="words")
words = words[[c for c in order if c in words.columns]]
words.fillna("").astype(str).replace(r"\.0$", "", regex=True)

In [ ]:
# Check 2: the safety check and other warnings.
CORE = ["1", "1A", "7", "8"]
problems = []
for filing, group in summary.groupby("filing"):
    found = set(group["item"])
    if "15" in found:
        found.add("8")             # the financial statements are in Item 15 (see step 1.2's notes)
    for item in CORE:
        if item not in found:
            problems.append((filing, item, "core Item missing"))
    for _, row in group.iterrows():
        if row["words"] >= 3000 and row["bold_headings"] < 3:
            problems.append((filing, row["item"], f"{row['words']} words but only {row['bold_headings']} bold headings"))
        if row["confidence"] < 0.8:
            problems.append((filing, row["item"], f"low confidence {row['confidence']:.2f} ({row['method']})"))
        # Raised from 40,000 on Nov 26: CRWV 1A (40,324), FIG 1A (39,158) and DELL 8 (45,987) were
        # checked by hand and are real long sections, each ending where it should.
        if row["words"] > 50000:
            problems.append((filing, row["item"], f"suspiciously long: {row['words']} words"))

print(len(problems), "problems")
pd.DataFrame(problems, columns=["filing", "item", "problem"])

In [ ]:
# Check 3: bold headings found in each core Item (step 1.3 builds the table of contents from these).
heads = summary[summary["item"].isin(CORE)].pivot(index="filing", columns="item", values="bold_headings")
heads[[c for c in CORE if c in heads.columns]]